---
title: "2.8 Dates and time series basics"
---

::: {.lesson-tools}
[Open in Colab](https://colab.research.google.com/github/ningzhanglab/Python-for-Beginners/blob/main/lessons/02-data/08-dates-time-series.ipynb){.btn .btn-primary}
[Download notebook](../../downloads/02-data/08-dates-time-series.ipynb){.btn .btn-outline-secondary}
:::

**Run in your browser.** Colab needs a Google account to run and save work. If its GitHub link cannot open, download this notebook and use **File → Upload notebook** at [Colab](https://colab.research.google.com/). You can read the website without an account. A notebook is a document containing explanatory text and runnable code cells.

**Time:** about 40 minutes, including exercises · **Before you begin:** Complete [2.7 Wide and long data; panel structure](../02-data/07-reshaping.ipynb) first. Each notebook starts afresh: it does not remember variables from another lesson.

## You will learn

- Parse date text with an explicit format and inspect invalid values.
- Sort one unit over time before differences and growth rates.
- Distinguish observation frequency, time bins and an annual growth rate.

## Why it matters

A **time series** is a measurement observed over time for one unit, such as one country. Dates can arrive as text; sorting text is not always sorting time. Our running panel contains several countries and observations every five years. A change between adjacent rows is meaningful only after choosing the unit, sorting dates and checking the gap. We begin with made-up survey dates, then follow Canada over the observed years.

## Worked example

### Read a date format explicitly

These fictional survey records use year-month-day text. `pd.to_datetime` converts it into pandas dates. `%Y` means a four-digit year, `%m` month and `%d` day. `.dt` gives access to date components such as the year.

In [ ]:
import pandas as pd
small = pd.DataFrame({"country": ["Northland", "Northland", "Eastria"], "date_text": ["2020-03-01", "2020-01-01", "2020-02-01"], "responses": [40, 30, 35]})
small["date"] = pd.to_datetime(small["date_text"], format="%Y-%m-%d")
small["year"] = small["date"].dt.year
small.sort_values("date")

The rows sort January, February, March. The counts are made-up numbers and the dates are invented examples. The format avoids ambiguity: text like 03/04/2020 could mean March 4 or April 3 depending on the source. Confirm the source convention rather than guessing.

### Inspect invalid dates

An impossible date should not silently become a plausible one. With `errors="coerce"`, invalid dates become **NaT**, pandas' missing-date marker. Count and retain the original strings.

In [ ]:
raw_dates = pd.Series(["2020-02-29", "2020-02-30"])
parsed_dates = pd.to_datetime(raw_dates, format="%Y-%m-%d", errors="coerce")
print(parsed_dates)
print("Invalid dates:", parsed_dates.isna().sum())

February 29 is valid in leap-year 2020; February 30 becomes NaT. One conversion failed. Coercion reports missingness; it is not a correction of the source or permission to drop the record without explanation.

### Prepare the real country panel

Run this prepared setup once in a fresh session. It uses the course CSV in a local checkout, or downloads its fixed, checksum-verified public source in Colab. A checksum is a fingerprint of a file's contents; a mismatch stops the download from being used. You do not need to retype the setup. It does not use your Google Drive or any private data.

In [ ]:
# Prepared setup: use the course CSV locally; fetch its pinned source in Colab.
from pathlib import Path
import hashlib
import io
from urllib.request import urlopen
import pandas as pd

data_file = Path("data/gapminder.csv")
if not data_file.exists():
    source = "https://raw.githubusercontent.com/jennybc/gapminder/391b5a40d3574c3b89d889dd7d72f7f3fdb4a70c/inst/extdata/gapminder.tsv"
    with urlopen(source, timeout=30) as response:
        raw = response.read()
    expected = "f74275b4a2f4a7c8e3a09a752f9d6e43d1759d78d5cbfdebaaa3e1a6efc68f78"
    if hashlib.sha256(raw).hexdigest() != expected:
        raise ValueError("The data download changed. Please report this to the course maintainer.")
    downloaded = pd.read_csv(io.BytesIO(raw), sep="\t", dtype=str)
    downloaded = downloaded.rename(columns={"continent": "region", "lifeExp": "life_exp", "pop": "population", "gdpPercap": "gdp_pc"})
    data_file.parent.mkdir(parents=True, exist_ok=True)
    downloaded.to_csv(data_file, index=False, lineterminator="\n")
print("Course data ready: data/gapminder.csv")

Look for **Course data ready**. Colab working files are temporary; rerun this cell after a runtime reset. The data cover 142 countries at five-year intervals from 1952 to 2007. Each row is one country in one year.

### Choose one country and an honest date label

The panel records calendar years, not interview days. Select Canada and sort the year column. We create January 1 labels as a convention for representing those years as dates. They do **not** identify the actual day on which GDP or life expectancy was measured.

In [ ]:
countries = pd.read_csv(data_file)
canada = countries.loc[countries["country"] == "Canada", ["year", "life_exp", "gdp_pc"]].sort_values("year").copy()
canada["date"] = pd.to_datetime(canada["year"].astype(str) + "-01-01", format="%Y-%m-%d")
print("Repeated years:", canada["year"].duplicated().sum())
print("Year gaps:", canada["year"].diff().dropna().unique().tolist())
canada.head()

No year repeats; all observed gaps are five years. `.diff()` subtracts the preceding value in order; the first result is missing because there is no earlier row. This is a 12-observation historical series, not annual measurements from 1952 to 2007.

### Compute changes between observed dates

Use `.set_index("date")` to make date labels the row index. Select the two measurements. `.pct_change(fill_method=None)` returns a fractional change from the preceding observation without filling gaps; multiply by 100 for percent. `.diff()` on life expectancy gives a difference in years.

In [ ]:
series = canada.set_index("date")[["life_exp", "gdp_pc"]].copy()
series["life_gain_years"] = series["life_exp"].diff()
series["gdp_growth_5y_pct"] = series["gdp_pc"].pct_change(fill_method=None) * 100
series.tail()

The final row compares 2007 with 2002: the life-expectancy gain is 0.883 years and GDP per person rises about 8.97 percent across five years. Sorting the entire multi-country panel and applying diff would mix units at country boundaries, so select one country or group by country first.

### Convert to an annualized growth rate

An **annualized rate** is the constant yearly rate that would reproduce the observed multi-year ratio under compounding. For a verified five-year gap, take the fifth root of the GDP ratio and subtract one. Python uses `**` for a power; the exponent `1 / 5` takes the fifth root. `.shift(1)` lines up each row with the preceding observed value.

In [ ]:
series["annualized_gdp_growth_pct"] = ((series["gdp_pc"] / series["gdp_pc"].shift(1)) ** (1 / 5) - 1) * 100
series[["gdp_growth_5y_pct", "annualized_gdp_growth_pct"]].tail(2)

The final annualized rate is about 1.73 percent. It is a summary rate, not five newly observed annual growth rates. Dividing a five-year percentage by five is only an approximation; compounding uses the ratio formula. If gaps differ, the exponent must use each verified gap, not an assumed constant.

### Group time into wider bins

`.resample` groups a date-indexed series into time bins. `10YS` requests ten-year bins starting at a calendar-year boundary. Count the observed points in each bin as well as their mean. This operation summarizes the measurements available in a bin; it does not interpolate missing annual observations.

In [ ]:
decade_bins = series["life_exp"].resample("10YS").agg(["mean", "count"])
decade_bins

Read the bin labels and counts together. Here each bin contains two five-year observations; it is not a mean of ten annual life-expectancy measurements. Keep the original series and state the bin convention whenever you report a time aggregation.

## Try it

Work in your own notebook copy. Add a new code cell for each task, try it yourself, then open the solution to compare. Run from the top if a name is missing.

### 1. Choose an unambiguous format

The text `"31/12/2020"` is explicitly day/month/year. Parse it with format `%d/%m/%Y` and print its year.

<details><summary>Solution 1</summary>

Run this complete solution after the worked example.

In [ ]:
day_month_date = pd.to_datetime("31/12/2020", format="%d/%m/%Y")
print(day_month_date.year)

The year is 2020. The format came from the stated convention, not from a guess based on a different locale.

</details>

### 2. Select a date interval

Select Canadian observations from January 1, 1990 onward using the date index and .loc. Save them as `recent_canada` and count them.

<details><summary>Solution 2</summary>

Run this complete solution after the worked example.

In [ ]:
recent_canada = series.loc[series.index >= pd.Timestamp("1990-01-01")]
print(len(recent_canada))
recent_canada[["life_exp", "gdp_pc"]]

There are four observations: 1992, 1997, 2002 and 2007. No observations for 1990 or 1991 have been invented. “Recent” is only a variable name within this historical series.

</details>

### 3. Check the last change

Retrieve the GDP and life-expectancy changes at the 2007 label. Describe the period and units for each.

<details><summary>Solution 3</summary>

Run this complete solution after the worked example.

In [ ]:
last_change = series.loc[pd.Timestamp("2007-01-01"), ["life_gain_years", "gdp_growth_5y_pct", "annualized_gdp_growth_pct"]]
print(last_change.round(3))

The life change is 0.883 years; GDP growth is about 8.972 percent across 2002–2007, or about 1.73 percent as an annualized compound rate. The January label represents the observation year, not an actual measurement day.

</details>

## When it goes wrong

The following examples deliberately trigger specific errors. The prepared `try`/`except` wrapper catches that named error and prints its message so the notebook can continue. It is for these demonstrations; an unexpected error elsewhere should still stop execution. You can read the message without memorizing the wrapper.

### An impossible date is treated as valid

Without coercion, an invalid calendar date raises an error.

In [ ]:
try:
    pd.to_datetime("2020-02-30", format="%Y-%m-%d")
except ValueError as error:
    print(f"{type(error).__name__}: {error}")

Check the raw text and source convention. If you use errors="coerce", count and review the NaT results; do not silently replace them with arbitrary dates.

### Text is mistaken for dates

The .dt accessor requires date-like values, not an unconverted text column.

In [ ]:
try:
    small["date_text"].dt.year
except AttributeError as error:
    print(f"{type(error).__name__}: {error}")

Convert with pd.to_datetime first. Formatting text to look like a date does not change its underlying type.

## Recap

- A time series needs a unit, ordered dates and a known frequency.
- Parse explicit source formats and count failed date conversions.
- A five-year change, an annualized rate and an observed annual rate are different quantities.
- Resampling summarizes observed measurements; it does not create missing data.

**Core path complete.** You can now inspect and reshape data. Continue to [3.1 First plots from a DataFrame](../03-viz/01-first-plots.ipynb). Review the [course map](../index.qmd) and rerun one data lesson from an empty session before moving on.

::: {.callout-note title="Coming from Stata / R / Excel?"}
Spreadsheet date displays can conceal whether a value is text or a date serial. In pandas, inspect the dtype and parse a stated format. Stata and R time-series operations also require attention to unit and time spacing.
:::